# Phase 4 - Experiment 6: hierarchical whole-case model (frozen encoder + chunk-level head) and section ablation

Experiments 1-5 read only the **last 1536 tokens**. Here the model reads the **whole case** (up to the last 16 x 1536
= 24,576 tokens, enough for 99% of cases):

```
case -> chunks of 1536 tokens (from the end backwards)
     -> FROZEN experiment-1 model (A2 fine-tuned) -> one 640-d vector per chunk   (stage 1, no training)
     -> + "distance from the end" embedding -> BiGRU over chunks -> attention over chunks -> Linear -> 0/1   (stage 2)
```

**Technique:** layer freezing / two-stage training: the whole 70M encoder is frozen, only the chunk head
(~0.3M parameters) is trained.

**Why train the head on dev (5-fold cross-validation)?** The encoder was fine-tuned on the training cases, so its
vectors for those cases already "know" the labels; a head trained on them would learn to trust the encoder blindly.
The encoder never saw dev, so: split dev into 5 folds, train a head on 4, predict the 5th -> every dev case gets an
honest (out-of-fold) prediction. For test, the fold heads are averaged. Each setting is repeated with 3 seeds
(dev is small, so we report mean ± std).

**Section ablation** (same head, same folds, different parts of the case):
`last` (only the last chunk = what experiment 1 reads) · `first` (only the first chunk: parties, facts, lower-court
history) · `first+last` · `all` (whole case).

Cached in `checkpoints/phase4/A2_hier_frozen/`. GPU time: only encoding dev + test (~16M tokens, a few minutes);
the heads train in seconds. It can run while `p4_run.py` is training (needs < 1 GB extra GPU memory).


In [1]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, cjpe_utils, cjpe_models, cjpe_experiment, cjpe_hier
for m in (cjpe_utils, cjpe_models, cjpe_experiment, cjpe_hier):
    importlib.reload(m)
from cjpe_experiment import Experiment, compare
import cjpe_utils as cu, cjpe_models as cm, cjpe_hier as ch

RUN = ROOT / "checkpoints" / "phase4" / "A2_hier_frozen"
RUN.mkdir(parents=True, exist_ok=True)
SEEDS = [0, 1, 2]
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## 1. Load the frozen experiment-1 model and encode every chunk of the dev and test cases (cached)


In [2]:
exp = Experiment(ROOT, {"phase1": "A2", "method": "full"})      # experiment 1: training is done -> just loads best_train.pt
model = exp._model()
for p in model.parameters():
    p.requires_grad_(False)
dev_v, dev_lg, dev_st = ch.encode_chunks(model, exp.ids["dev"], exp.device, exp.amp, RUN / "chunks_dev.npz", desc="dev chunks")
test_v, test_lg, test_st = ch.encode_chunks(model, exp.ids["test"], exp.device, exp.amp, RUN / "chunks_test.npz", desc="test chunks")
y_dev = exp.y["dev"]
n_ch = np.diff(dev_st)
print(f"dev: {len(n_ch)} cases, {n_ch.sum():,} chunks | chunks per case: mean {n_ch.mean():.1f}, median {int(np.median(n_ch))}, "
      f"max {n_ch.max()} | cases with 1 chunk {100 * (n_ch == 1).mean():.0f}%, >= 4 chunks {100 * (n_ch >= 4).mean():.0f}%")

# sanity check: the LAST chunk of every case is exactly the input of experiment 1 -> same logits as its dev_logits.npy
last_lg = dev_lg[dev_st[1:] - 1]
ref = np.load(exp.dir / "dev_logits.npy")
print("sanity: max |last-chunk logits - experiment-1 dev logits| =", float(np.abs(last_lg - ref).max()), "(should be ~0, bf16 noise)")

train: 37,387 records -> 31,652 unique cases | label-1 rate 0.4143 | tokens median 2,515, max 495,884
dev  : 1,988 records -> 994 unique cases | label-1 rate 0.5 | tokens median 3,386, max 52,650
test : 1,517 records -> 1,517 unique cases | label-1 rate None | tokens median 3,523, max 51,624
experiment: A2_full_causal_last1536_attn | device cuda | precision torch.bfloat16
parameters: 70.06M total, 70.06M trainable (100.0%)
class weights: [0.8537, 1.2069]
[cache] training 'train' already done -> loaded best_train.pt (best epoch 2)


test chunks: 100%|██████████| 656/656 [01:14<00:00,  8.76it/s]

dev: 994 cases, 3,375 chunks | chunks per case: mean 3.4, median 3, max 16 | cases with 1 chunk 11%, >= 4 chunks 34%
sanity: max |last-chunk logits - experiment-1 dev logits| = 0.0078125 (should be ~0, bf16 noise)


## 2. Baselines without any training: average / maximum of the encoder's own chunk predictions

The frozen encoder also gives a 0/1 score for every chunk. Simple whole-case rules: average P(1) over all chunks,
or use only the last chunk (= experiment 1).


In [3]:
p_chunk = cm.probs_1(dev_lg)
base = {}
base["last chunk only (= experiment 1)"] = cm.probs_1(last_lg)
base["mean P(1) over all chunks"] = np.array([p_chunk[a:b].mean() for a, b in zip(dev_st[:-1], dev_st[1:])])
base["mean P(1) over last 2 chunks"] = np.array([p_chunk[max(a, b - 2):b].mean() for a, b in zip(dev_st[:-1], dev_st[1:])])
rows = {k: {"F1 @0.5": ch.scores_with_threshold(y_dev, p)["at_0.5"]["macro_f1"],
            "F1 tuned": ch.scores_with_threshold(y_dev, p)["tuned"]["macro_f1"]} for k, p in base.items()}
display(pd.DataFrame(rows).T)

,F1 @0.5,F1 tuned
last chunk only (= experiment 1),0.7889,0.8099
mean P(1) over all chunks,0.7738,0.8087
mean P(1) over last 2 chunks,0.7909,0.8078


## 3. Section ablation with the trained chunk head (5-fold CV on dev x 3 seeds, cached)


In [4]:
abl_path = RUN / "section_ablation.json"
if abl_path.exists():
    ABL = json.load(open(abl_path))
    print("[cache] section ablation loaded")
else:
    ABL = {}
    for which in ["last", "first", "first+last", "all"]:
        seqs = ch.select_chunks(dev_v, dev_st, which)
        f1s, oofs = [], []
        for seed in SEEDS:
            t0 = time.time()
            oof, _ = ch.cross_validate(seqs, y_dev, exp.device, folds=5, seed=seed)
            f1s.append(ch.scores_with_threshold(y_dev, oof)["tuned"]["macro_f1"])
            oofs.append(oof)
        p = np.mean(oofs, 0)
        ABL[which] = {"F1 tuned per seed": f1s, "mean": round(float(np.mean(f1s)), 4), "std": round(float(np.std(f1s)), 4),
                      "seed-avg OOF F1 tuned": ch.scores_with_threshold(y_dev, p)["tuned"]["macro_f1"]}
        print(f"{which:11s}: dev macro-F1 (tuned) {np.mean(f1s):.4f} ± {np.std(f1s):.4f} | {time.time() - t0:.0f}s per seed")
    json.dump(ABL, open(abl_path, "w"), indent=1)
display(pd.DataFrame(ABL).T[["mean", "std", "seed-avg OOF F1 tuned"]].rename_axis("chunks used (dev, 5-fold CV)"))

last       : dev macro-F1 (tuned) 0.8092 ± 0.0008 | 19s per seed
first      : dev macro-F1 (tuned) 0.5966 ± 0.0081 | 19s per seed
first+last : dev macro-F1 (tuned) 0.8038 ± 0.0040 | 21s per seed
all        : dev macro-F1 (tuned) 0.8216 ± 0.0025 | 35s per seed


,mean,std,seed-avg OOF F1 tuned
"chunks used (dev, 5-fold CV)",,,
last,0.8092,0.0008,0.8175
first,0.5966,0.0081,0.6077
first+last,0.8038,0.004,0.8127
all,0.8216,0.0025,0.8318


## 4. Final model = `all` chunks: out-of-fold dev scores, error analysis by length, attention over the case


In [5]:
seqs_dev = ch.select_chunks(dev_v, dev_st, "all")
seqs_test = ch.select_chunks(test_v, test_st, "all")
final_path = RUN / "final.npz"
if final_path.exists():
    z = np.load(final_path)
    p_oof, p_test = z["p_oof"], z["p_test"]
    print("[cache] final predictions loaded")
else:
    oofs, heads = [], []
    for seed in SEEDS:
        oof, hs = ch.cross_validate(seqs_dev, y_dev, exp.device, folds=5, seed=seed)
        oofs.append(oof)
        heads += hs
    p_oof = np.mean(oofs, 0)
    p_test, attn_test = ch.predict_head(heads, seqs_test, exp.device, return_attn=True)
    _, attn_dev = ch.predict_head(heads, seqs_dev, exp.device, return_attn=True)   # (heads saw these cases: attention only, not scores)
    np.savez(final_path, p_oof=p_oof, p_test=p_test)
    torch.save([h.state_dict() for h in heads], RUN / "heads.pt")
    # attention by distance from the end, for cases with >= 4 chunks
    att = {}
    for a in attn_dev:
        if len(a) >= 4:
            for dist, w in enumerate(a[::-1][:8]):
                att.setdefault(dist, []).append(w)
    json.dump({str(k): round(float(np.mean(v)), 4) for k, v in att.items()}, open(RUN / "attention_by_distance.json", "w"), indent=1)

rep = ch.scores_with_threshold(y_dev, p_oof)
print("dev (out-of-fold) @0.5 :", rep["at_0.5"])
print(f"dev (out-of-fold) @{rep['threshold']:.2f}:", rep["tuned"])
pred = (p_oof >= rep["threshold"]).astype(int)
print("confusion:", cu.confusion(y_dev, pred))
n_tok = np.array([len(x) for x in exp.ids["dev"]])
p1 = cm.probs_1(ref)
thr1 = json.load(open(exp.dir / "dev_report.json"))["threshold"]
q = np.quantile(n_tok, [0, .25, .5, .75, 1])
rows = []
for a, b in zip(q[:-1], q[1:]):
    s = (n_tok >= a) & (n_tok <= b)
    rows.append({"tokens": f"{int(a)}-{int(b)}", "cases": int(s.sum()),
                 "hierarchical (all chunks)": cu.f1_scores(y_dev[s], pred[s])["macro_f1"],
                 "experiment 1 (last 1536)": cu.f1_scores(y_dev[s], (p1[s] >= thr1).astype(int))["macro_f1"]})
display(pd.DataFrame(rows).set_index("tokens"))
att = json.load(open(RUN / "attention_by_distance.json"))
print("mean attention weight by chunk position (0 = last chunk), cases with >= 4 chunks:", att)

dev (out-of-fold) @0.5 : {'macro_f1': 0.8219, 'macro_p': 0.8219, 'macro_r': 0.8219, 'f1_0': 0.8225, 'f1_1': 0.8214, 'acc': 0.8219, 'pred_1_rate': 0.497}
dev (out-of-fold) @0.32: {'macro_f1': 0.8318, 'macro_p': 0.8336, 'macro_r': 0.832, 'f1_0': 0.8259, 'f1_1': 0.8377, 'acc': 0.832, 'pred_1_rate': 0.5352}
confusion: {'TN': 396, 'FP': 101, 'FN': 66, 'TP': 431}


,cases,hierarchical (all chunks),experiment 1 (last 1536)
tokens,,,
188-2154,249,0.8053,0.8226
2154-3386,248,0.8461,0.8669
3386-5427,248,0.8305,0.7938
5427-52650,249,0.8434,0.7550


mean attention weight by chunk position (0 = last chunk), cases with >= 4 chunks: {'0': 0.4982, '1': 0.1757, '2': 0.129, '3': 0.0757, '4': 0.0552, '5': 0.0628, '6': 0.0466, '7': 0.0392}


## 5. Test submission (dev-tuned threshold) + log


In [6]:
test_ids = [d["id"] for d in exp.data["test"]]
zip_path = RUN / "submission.zip"
if not zip_path.exists():
    zip_path = cu.write_submission(test_ids, (p_test >= rep["threshold"]).astype(int), RUN)
cu.check_submission(zip_path, test_ids)

n_head = sum(p.numel() for p in ch.HierHead().parameters())
if not any(r.get("run") == "A2_hier_frozen" for r in cu.read_experiments(ROOT)):
    cu.log_experiment(ROOT, {"run": "A2_hier_frozen", "phase1": "A2", "method": "frozen encoder + chunk head (5-fold CV on dev)",
                             "input": "all chunks (<= 16 x 1536)", "pooling": "BiGRU + attention over chunks",
                             "trainable_M": round(n_head / 1e6, 3), "dev_f1_at_0.5": rep["at_0.5"]["macro_f1"],
                             "dev_f1_tuned": rep["tuned"]["macro_f1"], "threshold": rep["threshold"],
                             "note": "dev score is out-of-fold (5-fold CV x 3 seeds)", "date": time.strftime("%Y-%m-%d %H:%M")})
compare(ROOT)

submission OK: 1517 predictions, 874 x label 1 -> C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase4\A2_hier_frozen\submission.zip


,phase1,method,input,pooling,trainable_M,best_epoch,dev_f1_at_0.5,dev_f1_tuned,threshold,tokens_per_s,peak_gb
run,,,,,,,,,,,
A2_hier_frozen,A2,frozen encoder + chunk head (5-fold CV on dev),all chunks (<= 16 x 1536),BiGRU + attention over chunks,0.284,NaN,0.8219,0.8318,0.32,NaN,NaN
A2_full_causal_last1536_attn,A2,full,last1536,attn,70.060,2.0,0.7889,0.8099,0.33,89922.0,0.62
A_adapter64_causal_last1536_attn,A,adapter,last1536,attn,1.650,3.0,0.7578,0.7815,0.35,84299.0,0.53
A2_gradual_causal_last1536_attn,A2,gradual,last1536,attn,70.060,3.0,0.7362,0.7807,0.25,89924.0,0.62
B_lormar8_causal_last1536_attn,B,lorma,last1536,attn,1.070,3.0,0.7475,0.7807,0.33,60669.0,0.57
D_full_bidir_last1024_attn_kd2,D,full,last1024,attn,40.310,2.0,0.7228,0.7632,0.32,221304.0,0.37
D_full_bidir_last1024_attn,D,full,last1024,attn,40.310,2.0,0.7097,0.7390,0.43,216240.0,0.37
